-----------------------------------------------------------------------------------

## **Sistem Pemilihan Karyawan Berprestasi Untuk Bonus Tahunan**

###### **1. Abdillah Abhipraya Abrar 123240004**
###### **2. Putri Asmiranti 123240033**

----------------------------------------------

###### **Import Library**

In [1]:
import pandas as pd
import numpy as np

#### **Langkah 1: Memuat dan Membersihkan Dataset**
###### Pada tahap ini, mengimpor pustaka `pandas` dan `numpy`, membaca file data `salary_data.csv`, serta membersihkan baris yang kosong atau tidak lengkap (*null values*) agar tidak menyebabkan kesalahan perhitungan matematis.

In [2]:
# Membaca data dari file CSV
df = pd.read_csv('salary_data.csv') 

# Membersihkan data dari baris yang kosong murni atau mengandung nilai kosong (NaN)
df = df.dropna(how='all')
df = df.dropna()
df = df.reset_index(drop=True)

# Menampilkan jumlah baris data yang siap diproses
print(f"Jumlah baris setelah dibersihkan: {len(df)}")

Jumlah baris setelah dibersihkan: 373


##### **Langkah 2: Pra-pemrosesan Data (Transformasi Data Kategorikal)**
###### Rumus matematika algoritma WP hanya dapat memproses data berupa angka (*numerik*). Kolom gelar pendidikan seperti *Bachelor's*, *Master's*, atau *PhD* masih berupa teks (*kategorikal*).
###### Oleh karena itu, kita melakukan **Encoding/Mapping (Pemetaan)** menggunakan struktur *dictionary* di Python untuk menerjemahkan tingkat pendidikan menjadi angka bobot berjenjang (ordinal):
###### * **Bachelor's** diterjemahkan menjadi **1**
###### * **Master's** diterjemahkan menjadi **2**
###### * **PhD** diterjemahkan menjadi **3**
###### Hasil terjemahan ini kemudian disimpan ke dalam kolom baru bernama `Rating_Pendidikan`.

In [3]:
# Membuat aturan kamus terjemahan untuk gelar pendidikan
mapping_pendidikan = {
    "Bachelor's": 1, 
    "Master's": 2, 
    "PhD": 3
}

# Menerapkan kamus terjemahan ke kolom asli dan menyimpannya di kolom baru
df['Rating_Pendidikan'] = df['Education Level'].map(mapping_pendidikan)

# Menampilkan sampel data teratas pasca-pemetaan
df[['Education Level', 'Rating_Pendidikan']].head()

,Education Level,Rating_Pendidikan
0,Bachelor's,1
1,Master's,2
2,PhD,3
3,Bachelor's,1
4,Master's,2


##### **Langkah 3: Penentuan Bobot Kepentingan dan Pangkat Kriteria**
#### menentukan nilai tingkat kepentingan untuk 4 kriteria utama dengan skala nilai 1 sampai 5. Setiap kriteria juga diidentifikasi sifatnya:
###### 1. **Age (Umur):** Bobot 3, Sifat = *Benefit* (Makin senior/matang dinilai makin positif).
###### 2. **Pendidikan:** Bobot 4, Sifat = *Benefit* (Pendidikan lebih tinggi dinilai lebih baik).
###### 3. **Pengalaman Kerja:** Bobot 5, Sifat = *Benefit* (Masa kerja lama dinilai lebih ahli).
###### 4. **Salary (Gaji):** Bobot 2, Sifat = *Cost* (Gaji saat ini dinilai sebagai pengeluaran/beban bagi perusahaan, sehingga nilai gaji yang lebih rendah diprioritaskan untuk didorong lewat bonus agar adil).

#### **Aturan Normalisasi Pangkat WP:**
###### Total seluruh bobot harus bernilai **1** ($\sum W = 1$). Kriteria bersifit *Benefit* akan mendapatkan nilai pangkat **positif (+)**, sedangkan kriteria bersifat *Cost* mendapatkan nilai pangkat **negatif (-)**.

In [4]:
# Menentukan nilai bobot awal dan sifat masing-masing kriteria
bobot_kepentingan = [3, 4, 5, 2] 
sifat_kriteria = ['benefit', 'benefit', 'benefit', 'cost']

# Menghitung total bobot awal untuk pembagi normalisasi
total_bobot = sum(bobot_kepentingan)
w_normal = [b / total_bobot for b in bobot_kepentingan]

# Menentukan tanda pangkat (+ / -) berdasarkan sifat kriterianya
w_pangkat = []
for i, sifat in enumerate(sifat_kriteria):
    if sifat == 'benefit':
        w_pangkat.append(w_normal[i])
    else:
        w_pangkat.append(-w_normal[i])

print(f"Bobot Pangkat Hasil Normalisasi (W): {w_pangkat}")

Bobot Pangkat Hasil Normalisasi (W): [0.21428571428571427, 0.2857142857142857, 0.35714285714285715, -0.14285714285714285]


#### **Langkah 4: Perhitungan Vektor S**
###### Vektor S ($S_i$) dihitung untuk setiap individu karyawan. Caranya adalah dengan mengalikan seluruh nilai kriteria karyawan tersebut setelah masing-masing kriterianya dipangkatkan dengan bobot kriteria ternormalisasi yang sesuai.
###### Fungsi `max(value, 0.0001)` disisipkan untuk mengamankan rumus matematika agar tidak terjadi error pembagian dengan angka nol murni (*division by zero error*).

In [5]:
vektor_s = []

# Melakukan perulangan kalkulasi untuk setiap baris data karyawan
for index, row in df.iterrows():
    s_i = (max(row['Age'], 0.0001) ** w_pangkat[0]) * \
          (max(row['Rating_Pendidikan'], 0.0001) ** w_pangkat[1]) * \
          (max(row['Years of Experience'], 0.0001) ** w_pangkat[2]) * \
          (max(row['Salary'], 0.0001) ** w_pangkat[3])
    vektor_s.append(s_i)

# Menyimpan hasil kalkulasi vektor S ke tabel utama
df['Vektor_S'] = vektor_s

#### **Langkah 5: Perhitungan Vektor V dan Perankingan (Ranking)**
###### Vektor V ($V_i$) merupakan skor akhir preferensi atau nilai mutlak kelayakan karyawan. Nilai ini diperoleh dengan membagi nilai Vektor S milik individu tersebut dengan total keseluruhan jumlah nilai Vektor S dari semua karyawan di perusahaan. Setelah skor Vektor V didapatkan, data diurutkan dari yang terbesar ke terkecil untuk menentukan peringkat performa karyawan.

In [6]:
# Menghitung total nilai seluruh vektor S sebagai pembagi
total_s = sum(vektor_s)

# Menghitung nilai Vektor V (skor akhir preferensi)
df['Vektor_V'] = df['Vektor_S'] / total_s

# Mengurutkan data berdasarkan skor Vektor V tertinggi ke terendah
df_ranking = df.sort_values(by='Vektor_V', ascending=False).reset_index(drop=True)
df_ranking.index += 1 # Mengubah tampilan index agar dimulai dari peringkat 1

##### **Langkah 6: Eksekusi Keputusan Berbasis Ambang Batas & Alokasi Bonus Proporsional**
###### Perusahaan menerapkan kebijakan **Ambang Batas (Threshold)** nilai minimal sebesar **0.00400** sebagai syarat kelayakan mendapatkan bonus. 

###### Sistem kemudian membagi total dana anggaran bonus perusahaan sebesar **$50.000** secara **proporsional** hanya kepada karyawan yang lolos ambang batas tersebut. Karyawan dengan skor Vektor V yang lebih tinggi akan otomatis mengantongi porsi nominal bonus yang lebih besar.

In [7]:
# Menetapkan parameter acuan kebijakan HRD perusahaan
ambang_batas = 0.00400
anggaran_total = 50000

# Menentukan status kelulusan berdasarkan nilai ambang batas kaku (threshold)
df_ranking["Status Keputusan"] = ["Mendapat Bonus" if v >= ambang_batas else "Tidak Mendapat Bonus" for v in df_ranking["Vektor_V"]]

# Menghitung total akumulasi skor khusus milik kelompok karyawan yang dinyatakan lolos
total_v_penerima = df_ranking[df_ranking["Status Keputusan"] == "Mendapat Bonus"]["Vektor_V"].sum()

# Melakukan pembagian dana anggaran bonus secara proporsional sesuai kekuatan skor masing-masing
bonus_list = []
for _, row in df_ranking.iterrows():
    if row["Status Keputusan"] == "Mendapat Bonus" and total_v_penerima > 0:
        # Rumus distribusi proporsional dana
        bonus = (row["Vektor_V"] / total_v_penerima) * anggaran_total
        bonus_list.append(bonus)
    else:
        bonus_list.append(0)

# Memasukkan nominal hasil bagi dana ke kolom tabel hasil akhir
df_ranking["Nilai Bonus Diterima"] = bonus_list

# Menampilkan resume laporan singkat hasil eksekusi sistem keputusan
print("--- RESUME LAPORAN KEPUTUSAN BONUS KARYAWAN ---")
print(f"Standar Batas Nilai Kelulusan (V) : {ambang_batas}")
print(f"Pagu Anggaran Dana Tersedia      : ${anggaran_total:,}")
print(f"Jumlah Karyawan Lolos Kualifikasi : {len(df_ranking[df_ranking['Status Keputusan'] == 'Mendapat Bonus'])} Orang\n")

# Menampilkan output tabel hasil keputusan final (Top 15 Teratas)
kolom_tampil = ['Job Title', 'Education Level', 'Years of Experience', 'Salary', 'Vektor_V', 'Status Keputusan', 'Nilai Bonus Diterima']
display(df_ranking[kolom_tampil].head(15))

--- RESUME LAPORAN KEPUTUSAN BONUS KARYAWAN ---
Standar Batas Nilai Kelulusan (V) : 0.004
Pagu Anggaran Dana Tersedia      : $50,000
Jumlah Karyawan Lolos Kualifikasi : 29 Orang



,Job Title,Education Level,Years of Experience,Salary,Vektor_V,Status Keputusan,Nilai Bonus Diterima
1,Director of Human Resources,PhD,23.0,185000.0,0.004543,Mendapat Bonus,1846.835455
2,Research Scientist,PhD,22.0,160000.0,0.004486,Mendapat Bonus,1823.642422
3,Senior Research Scientist,PhD,22.0,160000.0,0.004486,Mendapat Bonus,1823.642422
4,Director of Operations,PhD,22.0,180000.0,0.004470,Mendapat Bonus,1817.148836
5,Director of Operations,PhD,22.0,180000.0,0.004470,Mendapat Bonus,1817.148836
6,Director of Sales and Marketing,PhD,22.0,180000.0,0.004470,Mendapat Bonus,1817.148836
7,Chief Technology Officer,PhD,24.0,250000.0,0.004437,Mendapat Bonus,1803.662783
8,Director of Operations,PhD,21.0,180000.0,0.004377,Mendapat Bonus,1779.487241
9,Senior Data Scientist,PhD,21.0,180000.0,0.004338,Mendapat Bonus,1763.667403
10,Principal Engineer,PhD,20.0,170000.0,0.004318,Mendapat Bonus,1755.313137
